# Sparse-scaling benchmark: Te

Trigonal Te (example11), a chiral semiconductor with spin-orbit coupling, three atoms and
`nawf = 54` per primitive cell (the ladder runs to d = 4, `nawf = 864`, for now). Te has no inversion centre, so its
bands are spin split and carry a spin texture. It tests the Rashba-Edelstein response, through
the spin and orbital textures (`rashba_edelstein`) and by the Kubo route
(`linear_response(response='ree')`), `spin_texture` on the mesh, and transport in an
anisotropic semiconductor. The PAO shift is 0.47 eV, just above the conduction-band edge, so the
properties stay at the valence-band top (−0.5 to 0 eV for the Rashba-Edelstein response).

A cell doubled `d` times is still the same crystal, so every property per primitive cell must
look the same at every size. Each figure shows one sparse pipeline. The colored lines are its
sizes, and the dashed black line is the dense reference on the primitive cell (`reference_d00`).
The curves come from [observables.py](../../observables.py), which normalizes them per primitive
cell, and the figures from [plots.py](../../plots.py). The settings are in
[system.py](system.py). A pipeline that does not run a property has no panel for it.

**Before the first ladder** there are no measured notes here yet. The expectations below come
from a smoke test at a 12 × 12 × 8 mesh, d ≤ 1 (2026-10-26):

- With the exact H(R), sparse d = 0 matches the dense reference to 10⁻¹² in the Kubo Rashba-Edelstein response, 10⁻¹³ in the Kubo σ and 10⁻⁴ in the Boltzmann σ; S differs by 1–3%, in the gap. The band-curvature family differs at d = 0 already: Hall 1–15%, Nernst 5–12%, masses 0.5–0.6. The band-diagonal `rashba_edelstein` and the spin texture were not measured against the reference with the exact H(R).
- d = 1 matches d = 0 to 10⁻¹¹ in σ, S and `cond`, and the DoS to 2×10⁻⁷ above −14 eV (the 4d peak below changes with the adaptive widths of its degenerate states, which is why the DoS starts at −14 eV). **The band-diagonal Rashba-Edelstein response moves by 7–27% per doubling** (spin xx 8%, spin zz 26%, orbital 7–11%), through the whole range. A likely cause, not checked state by state: doubling along x makes k and −k exactly degenerate on the planes k_x = ±1/4, and `texture_k` takes ⟨S⟩ from the raw eigenvectors while the velocities are resolved inside each degenerate group, so ⟨S⟩ and v belong to different states there.
- The components point group 32 zeroes are not small at this mesh, in the dense reference as in the sparse runs: REE xy is 14–20% of xx, and the Hall and Nernst components with a repeated index are of the order of the allowed ones (section on vanishing components).

In [ ]:
import os
import sys

# the shared modules live two levels up, in benchmarks/sparse_scaling
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from plots import Benchmark

bench = Benchmark('te', root=os.path.abspath('runs'), symlog=True)
show = bench.show

## 1. Density of states

The DoS per primitive cell, from −14 eV: the 4d semicore states (−38 eV) are solved but not
plotted. The PDOS is not compared (spinor copies share a file prefix, see the Pt notebook).

In [ ]:
show(['dos'], ncol=1)

## 2. Band structure at d = 0

Doubling folds the band path, so only the undoubled cell is compared: the dense reference against
`sparse` d = 0, on PAOFLOW's default hexagonal path. The sparse path holds the
well-projected bands, and NaN above the usable top (the PAO shift minus 0.2 eV). The right panel
is the difference per band within ±3 eV of E_F. `collect.py` reports the RMS over every band as
`dev_bands_vs_ref`, in eV.

Measured with the band path alone (2026-10-26): with the exact H(R) the sparse bands lie within 35
meV of the dense ones within 1 eV of E_F (9 meV RMS) and 52 meV anywhere, which is the off-grid
interpolation convention on the coarse 6 × 6 × 4 QE grid. The largest shell cutoff the QE grid
allows (47 shells) would move them by up to 110 meV near E_F and 110 meV elsewhere, which is why
Te runs without one.

In [ ]:
bench.show_bands()

## 3. Boltzmann transport

Constant relaxation time, at 300 K, against the chemical potential. The Hall tensor trace needs
the band curvature, so only `sparse_curvature` has it.

In [ ]:
show(['sigma_xx', 'sigma_zz', 'seebeck_xx', 'kappa_xx', 'pf_xx', 'hall_trace'])

## 4. Hall and Nernst tensors

The Hall components with three different indices, and two Nernst components. Both need the band
curvature, so only `sparse_curvature` has them.

In [ ]:
show(['hall_xyz', 'hall_xzy', 'hall_yzx', 'nernst_xyz', 'nernst_yxz'])

## 5. Band-diagonal and Kubo conductivity

`conductivity` sums v_i v_j over states with adaptive smearing; `linear_response(response='cond')`
is the Kubo formula with a Lorentzian broadening of 0.05 eV, which sums over every pair of
states, so only `sparse_curvature` runs it. Te is trigonal, so xx = yy differs from zz.

In [ ]:
show(['cond_xx', 'cond_zz', 'lr_cond_xx', 'lr_cond_zz'], ncol=4)

## 6. Rashba-Edelstein response

Point group 32 allows the diagonal xx = yy and zz. `rashba_edelstein` is band-diagonal: it sums
the spin (orbital) texture times the band velocity with adaptive smearing, and the curve is the
accumulation over the charge current, a ratio. `linear_response(response='ree')` is the Kubo
route with a 0.05 eV broadening; it sums over every pair of states, so only `sparse_curvature`
runs it.

In [ ]:
show(
    [
        'ree_spin_xx',
        'ree_spin_zz',
        'ree_orbital_xx',
        'ree_orbital_zz',
        'lr_ree_even_xx',
        'lr_ree_even_zz',
    ],
    ncol=2,
)

## 7. Spin texture (d = 0 only)

`spin_texture` writes ⟨S⟩ on the mesh, one file per band of the window. Doubling folds the bands,
so only the undoubled cell is compared, as the RMS of |⟨S⟩| over the mesh for each band.

In [ ]:
show(['spin_texture'], ncol=1)

## 8. Effective masses

Smeared densities per k-point of the primitive cell, of the inverse DoS mass and of the mean
inverse principal mass (see the Si notebook for why only these symmetric functions compare).

In [ ]:
show(['inv_mass_dos', 'inv_mass_trace'], ncol=2)

## 9. Components that must vanish

Point group 32 zeroes the off-diagonal Boltzmann and conductivity components, the Hall and Nernst
components with a repeated index, and the off-diagonal xy of the Rashba-Edelstein response.

In [ ]:
show(
    [
        'sigma_xy',
        'sigma_xz',
        'sigma_yz',
        'seebeck_xy',
        'kappa_xy',
        'pf_xy',
        'hall_zero',
        'nernst_zero',
        'cond_xy',
        'lr_cond_xy',
        'ree_spin_xy',
        'ree_orbital_xy',
        'lr_ree_even_xy',
    ],
    normalized=True,
)

## 10. What is not tested here

- **`dielectric_tensor`**: the conduction bands it needs lie above the PAO shift and are not
  projectable.
- **`doping`**: `solve_for_mu` weights the DoS by 2 for spin, which double counts the spinor
  bands of a spin-orbit run, so μ lands in the 4d shell (−36 eV), in the dense pipeline as in the
  sparse one.
- **The spin and orbital Hall and the anomalous Hall conductivities**: Pt and Fe test them.
- **PDOS**: spinor copies share a file prefix (see the Pt notebook).